In [2]:
!pip install -q transformers sentence-transformers razdel sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 19.5 MB/s eta 0:00:00


# Генерация train_backtranslate.csv

In [3]:
import re
import random
import warnings
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

warnings.filterwarnings('ignore')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

device: cuda


In [5]:
from google.colab import drive
import os

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/papadyk-collab/vkr'
output_dir = os.path.join(drive_root, 'output')
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(drive_root, 'train.csv')
out_path = os.path.join(output_dir, 'backtranslate')

Mounted at /content/drive


In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

In [7]:
import pandas as pd

df = pd.read_csv(train_path)
counts = df["label"].value_counts()
small_labels = counts[counts < 30].sort_values(ascending=True).index
df_small = df[df["label"].isin(small_labels)]

for label in small_labels:
    print("=" * 80)
    print(f"LABEL: {label}")
    print("=" * 80)

    texts = df_small.loc[df_small["label"] == label, "text"]
    for i, t in enumerate(texts, start=1):
        print(f"\n--- sample {i} ---\n")
        print(t)

    print("\n\n")

LABEL: Имущественные вопросы

--- sample 1 ---

[ORGANIZATION] ПРИКАЗ [DATE_TIME] No [DOCUMENT_NUMBER] Об утверждении Положений по непрофильным активам и порядке отчуждения непрофильных активов В целях приведения локальных нормативных актов Группы [ORGANIZATION] по вопросам управления непрофильными активами в соответствии с нормативными актами [ORGANIZATION] ПРИКАЗЫВАЮ: 1. Утвердить Положение о Комиссии по непрофильным активам [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 1). 2. Утвердить Положение о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 2). 3. Утвердить состав Комиссии по непрофильным активам [ORGANIZATION] в новом составе (Приложение 3). 4. Признать утратившими силу приказы [ORGANIZATION] от [DATE_TIME] No [DOCUMENT_NUMBER] и от [DATE_TIME] No [DOCUMENT_NUMBER]. 5. Распространить действие настоящего Приказа на организации Группы компаний [ORGANIZATION]

In [8]:
import torch
import re
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util
from razdel import sentenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- Модели перевода (Helsinki-NLP) ---
ru_en_name = "Helsinki-NLP/opus-mt-ru-en"
en_ru_name = "Helsinki-NLP/opus-mt-en-ru"

ru_en_tokenizer = MarianTokenizer.from_pretrained(ru_en_name)
ru_en_model = MarianMTModel.from_pretrained(ru_en_name).to(device)

en_ru_tokenizer = MarianTokenizer.from_pretrained(en_ru_name)
en_ru_model = MarianMTModel.from_pretrained(en_ru_name).to(device)

# --- Эмбеддинги ---
embed_model = SentenceTransformer("deepvk/USER-base", device=device)

def translate_block(text: str, tokenizer, model, max_length=400) -> str:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    ).to(device)
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

def clean_bt_result(text: str) -> str:
    text = re.sub(r'([,.!?])\1{2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r',\s*', ', ', text)
    return text.strip()

def mask_placeholders(text: str) -> tuple[str, dict]:
    """
    Заменяем [ORGANIZATION], [DATE_TIME] и т.п. на MASK_0, MASK_1, ...
    """
    pattern = r'\[[^\]]+\]'
    mapping = {}
    counter = [0]

    def replacer(m):
        token = m.group(0)
        key = f"MASK_{counter[0]}"
        mapping[key] = token
        counter[0] += 1
        return key

    masked = re.sub(pattern, replacer, text)
    return masked, mapping

def unmask_placeholders(text: str, mapping: dict) -> str:
    for key, original in mapping.items():
        text = text.replace(key, original)
    return text

def back_translate_aggressive_one(text_orig: str) -> str:
    """Back-translation одного документа с разбиением по предложениям + маскировкой плейсхолдеров."""
    # 1. Маскируем плейсхолдеры
    masked_text, mapping = mask_placeholders(text_orig)

    # 2. Разбиение на предложения
    sentences = [s.text.strip() for s in sentenize(masked_text) if s.text.strip()]
    bt_sentences = []

    for s in sentences:
        if len(s) > 80:
            try:
                en = translate_block(s[:400], ru_en_tokenizer, ru_en_model, max_length=400)
                ru_bt = translate_block(en, en_ru_tokenizer, en_ru_model, max_length=400)
                ru_bt = clean_bt_result(ru_bt)
                bt_sentences.append(ru_bt)
            except Exception as e:
                print(f"Ошибка BT: {e}")
                bt_sentences.append(s)
        else:
            bt_sentences.append(s)

    result_masked = " ".join(bt_sentences)

    # 3. Восстанавливаем плейсхолдеры
    result = unmask_placeholders(result_masked, mapping)
    return result

# -------- Тест на одной строке --------
source_text = "[ORGANIZATION] Почтовый/Юридический адрес: [LOCATION] Телефон: [CONTACT]; [CONTACT], факс: [CONTACT] e-mail: [CONTACT] ОКПО [ID], ОГРН [ID], ИНН/КПП [ID]/[ID] от [DATE_TIME] No [DOCUMENT_NUMBER] [DOCUMENT_NUMBER] [DATE_TIME] г. на No от Руководителю проекта [OBJECT] [PERSON] [CONTACT] О проезде техники через КПП [OBJECT] Уважаемый [PERSON]! На письмо исх. No [DOCUMENT_NUMBER] от [DATE_TIME] сообщаем Вам, что [ORGANIZATION] не согласовывает проезд транспорта [ORGANIZATION] через ДКП-1 с проездом по территории [OBJECT] и выездом через ДКП-2 ([NUMBER] км) к трассе трубопровода [ORGANIZATION]. Рекомендуем Вам рассмотреть проезд по вдольтрассовому проезду (ВТП) [ORGANIZATION] с заездом через ДКП-3 ([NUMBER] км), а/т должен быть оснащен искрогасителями, также персонал иметь при себе подтверждающий документ о прохождении инструктажа по требованиям пожарной безопасности. В случае Вашего согласия просим подтвердить официальным письмом. Обращаем Ваше внимание, что в мае-июне на ВТП [ORGANIZATION] будут действовать сезонные ограничения по проезду техники, связанные с паводковым периодом в [LOCATION], просим Вас заблаговременно уточнять в [ORGANIZATION] о возможности проезда по ВТП. Для оформления пропусков просим Вас направить [PERSON], [PERSON], наименование перевозимого груза с привязкой к автомобильному транспорту в форме заявки (приложение), а также лист ознакомления со Стандартом [ORGANIZATION] пропускной и внутритобъектовый режимы на территории производственных и иных объектов No [DOCUMENT_NUMBER]. Приложение: Образец заявки на проезд – на 1 л. в 1 экз. Первый заместитель генерального директора по производству – главный инженер [PERSON] [PERSON] [CONTACT], доб. [NUMBER] [CONTACT] ДОКУМЕНТ ПОДЛИННАЯ ЭЛЕКТРОННОЙ ПОДЛИННОСТИ Сертификат [FINANCIAL_DATA] [FINANCIAL_DATA] Владелец [PERSON] Действителен с [DATE_TIME] по [DATE_TIME]"

print(f"Оригинал: {len(source_text)} символов")
print(source_text[:400] + "..." if len(source_text) > 400 else source_text)

bt_text = back_translate_aggressive_one(source_text)

print(f"\n=== BACK-TRANSLATION (2-й трек) ===")
print(f"Результат: {len(bt_text)} символов")
print(bt_text[:400] + "..." if len(bt_text) > 400 else bt_text)

orig_emb = embed_model.encode(source_text, convert_to_tensor=True)
bt_emb = embed_model.encode(bt_text, convert_to_tensor=True)
sim = util.cos_sim(orig_emb, bt_emb).item()
print(f"\nCosine similarity: {sim:.3f}")

Device: cuda


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Оригинал: 1851 символов
[ORGANIZATION] Почтовый/Юридический адрес: [LOCATION] Телефон: [CONTACT]; [CONTACT], факс: [CONTACT] e-mail: [CONTACT] ОКПО [ID], ОГРН [ID], ИНН/КПП [ID]/[ID] от [DATE_TIME] No [DOCUMENT_NUMBER] [DOCUMENT_NUMBER] [DATE_TIME] г. на No от Руководителю проекта [OBJECT] [PERSON] [CONTACT] О проезде техники через КПП [OBJECT] Уважаемый [PERSON]! На письмо исх. No [DOCUMENT_NUMBER] от [DATE_TIME] сообща...

=== BACK-TRANSLATION (2-й трек) ===
Результат: 1659 символов
[ORGANIZATION] Почтовый/Юридический адрес: [LOCATION] Телефон: [CONTACT]; [CONTACT], факс: [CONTACT] электронная почта: [CONTACT] [ID], [ID], INN/[ID]/[ID] от [LOCATION]0 No. [LOCATION]1 [LOCATION]2 [LOCATION]3 до [LOCATION]4 [LOCATION]5 [LOCATION]6 Руководитель проекта по прохождению оборудования через контрольно-пропускной пункт [LOCATION]7, уважаемый MASC_18. На письмо исх. No. MASC_19 от [CONT...

Cosine similarity: 0.850


In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from razdel import sentenize
import torch
import re
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ====== МОДЕЛИ ======
# эмбеддинги
embed_model = SentenceTransformer("deepvk/USER-base", device=device)

# перевод ru↔en (Helsinki-NLP)
ru_en_name = "Helsinki-NLP/opus-mt-ru-en"
en_ru_name = "Helsinki-NLP/opus-mt-en-ru"

ru_en_tokenizer = MarianTokenizer.from_pretrained(ru_en_name)
ru_en_model = MarianMTModel.from_pretrained(ru_en_name).to(device)

en_ru_tokenizer = MarianTokenizer.from_pretrained(en_ru_name)
en_ru_model = MarianMTModel.from_pretrained(en_ru_name).to(device)

# ====== НАСТРОЙКИ ======
SIM_MIN_BT = 0.7
SIM_MAX_BT = 0.9
TARGET_PER_CLASS = 30

def translate_block(text: str, tokenizer, model, max_length=400) -> str:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    ).to(device)
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

def clean_bt_result(text: str) -> str:
    text = re.sub(r'([,.!?])\1{2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r',\s*', ', ', text)
    return text.strip()

def back_translate_aggressive_one(text_orig: str) -> str:
    """Back-translation одного документа с разбиением на предложения."""
    sentences = [s.text.strip() for s in sentenize(text_orig) if s.text.strip()]
    bt_sentences = []

    for s in sentences:
        if len(s) > 80:
            try:
                en = translate_block(s[:400], ru_en_tokenizer, ru_en_model, max_length=400)
                ru_bt = translate_block(en, en_ru_tokenizer, en_ru_model, max_length=400)
                ru_bt = clean_bt_result(ru_bt)
                bt_sentences.append(ru_bt)
            except Exception as e:
                print(f"Ошибка BT: {e}")
                bt_sentences.append(s)
        else:
            bt_sentences.append(s)

    return " ".join(bt_sentences)

aug_rows = []

aug_file_path = os.path.join(out_path, "train_backtranslate_partial.csv")
if os.path.exists(aug_file_path):
    df_prev = pd.read_csv(aug_file_path)
    aug_rows = df_prev.to_dict(orient="records")
    print(f"Загружено уже сгенерированных BT-примеров: {len(aug_rows)}")

label_to_texts = {
    label: df_small.loc[df_small["label"] == label, "text"].tolist()
    for label in small_labels
}

for label in tqdm(small_labels, desc="BT labels"):
    texts_orig = label_to_texts[label]
    current_count = len(texts_orig)

    already_bt = [r for r in aug_rows if r["label"] == label]
    current_count_with_bt = current_count + len(already_bt)

    if current_count_with_bt >= TARGET_PER_CLASS:
        continue

    need = TARGET_PER_CLASS - current_count_with_bt
    print(f"\nLabel: {label} | есть {current_count_with_bt}, нужно добить BT: {need}")

    orig_embeddings = embed_model.encode(texts_orig, convert_to_tensor=True)

    orig_idx = 0
    attempts = 0
    max_attempts = need * 20

    while need > 0 and attempts < max_attempts:
        attempts += 1
        text_orig = texts_orig[orig_idx]
        orig_emb = orig_embeddings[orig_idx]

        # --- back-translation ---
        bt_text = back_translate_aggressive_one(text_orig)

        # --- cosine similarity ---
        emb_bt = embed_model.encode(bt_text, convert_to_tensor=True)
        sim = util.cos_sim(orig_emb, emb_bt).item()

        if SIM_MIN_BT <= sim <= SIM_MAX_BT:
            aug_rows.append({
                "label": label,
                "text": bt_text,
                "source_text": text_orig,
                "cosine_sim": sim,
            })
            need -= 1

            if len(aug_rows) % 5 == 0:
                df_aug_partial = pd.DataFrame(aug_rows)
                df_aug_partial.to_csv(aug_file_path, index=False)
                print(f"BT: сохранено {len(aug_rows)} аугментированных примеров в {aug_file_path}")

        orig_idx = (orig_idx + 1) % len(texts_orig)

# финальное сохранение BT-аугментаций
df_bt = pd.DataFrame(aug_rows)
df_bt.to_csv(aug_file_path, index=False)
print(f"\nИтого BT-аугментированных примеров: {len(df_bt)}")
print(f"Промежуточный файл сохранён в: {aug_file_path}")

# склейка с исходным df
df_full_bt = pd.concat([df, df_bt[["label", "text"]]], ignore_index=True)
final_path = os.path.join(out_path, "train_backtranslate.csv")
df_full_bt.to_csv(final_path, index=False)
print(f"Финальный датасет (оригинал + BT) сохранён в: {final_path}")

Device: cuda


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Загружено уже сгенерированных BT-примеров: 397


BT labels:   0%|          | 0/23 [00:00<?, ?it/s]


Label: Проект «Трубопроводный транспорт Ещё одного НГКМ» | есть 1, нужно добить BT: 29
